# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant
# Visualization
!pip install matplotlib seaborn

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
from pprint import pprint
import warnings
warnings.filterwarnings('ignore')

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print basic summary
print(f"{metadata.name}: {metadata.description}")
print(f"Published: {metadata.datePublished}")
print(f"License: {metadata.license}")
print(f"Version: {metadata.version}")
print(f"Keywords: {metadata.keywords}")


## 2. Data Overview
Review available record sets, fields, and their IDs. All entities are referenced by their `@id` values.

In [ ]:
# List available record sets with their @id
print("Available record sets in this dataset:")
for rs in metadata.record_sets:
    print(f"RecordSet @id: {rs.id}, Name: {rs.name}")

# For each record set, summarize the available fields
record_sets_summary = {}
for rs in metadata.record_sets:
    print(f"\nRecordSet @id: {rs.id}")
    print(f"  Name: {rs.name}")
    print(f"  Description: {rs.description}")
    print("  Fields:")
    fields_out = []
    for fld in rs.fields:
        print(f"    Field @id: {fld.id}, name: {fld.name}, type: {fld.data_type}")
        fields_out.append({
            "id": fld.id, "name": fld.name, "data_type": fld.data_type
        })
    record_sets_summary[rs.id] = fields_out

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set
# For this dataset, we'll discover which record sets are available and extract all. 

import collections
dataframes = {}
record_set_ids = [rs.id for rs in metadata.record_sets]

if not record_set_ids:
    print("No record sets found in metadata, please check Croissant definition.")
else:
    for record_set_id in record_set_ids:
        print(f"Loading records for RecordSet @id: {record_set_id}")
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Columns for RecordSet {record_set_id}:\n{df.columns.tolist()}")
            print(df.head())
        else:
            print(f"No records found in record set '{record_set_id}'")

# Set one record set as example for EDA
if record_set_ids:
    main_record_set_id = record_set_ids[0]
    print(f"\nProceeding with {main_record_set_id} for further analysis.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# If dataframe exists, run EDA

if 'main_record_set_id' in locals() and main_record_set_id in dataframes:
    df = dataframes[main_record_set_id]
    print(f"Head of DataFrame for {main_record_set_id}:")
    print(df.head())
    
    # Find a numeric field for filtering and normalization
    numeric_field_id = None
    for fld in record_sets_summary.get(main_record_set_id, []):
        if fld['data_type'] in ['Float', 'Number', 'Integer'] and fld['id'] in df.columns:
            numeric_field_id = fld['id']
            break
    
    if numeric_field_id is None:
        print("No numeric field found for EDA.")
    else:
        print(f"Using numeric field: {numeric_field_id}")
        if pd.api.types.is_numeric_dtype(df[numeric_field_id]):
            threshold = df[numeric_field_id].mean()  # Use mean as threshold
            filtered_df = df[df[numeric_field_id] > threshold].copy()
            print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
            print(filtered_df.head())
            # Normalize
            filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
            print(f"Normalized {numeric_field_id} for filtered records:")
            print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
            # Group by first categorical/text field
            group_field_id = None
            for fld in record_sets_summary.get(main_record_set_id, []):
                if (fld['data_type'] in ['Text', 'String']) and fld['id'] in df.columns:
                    group_field_id = fld['id']
                    break
            if group_field_id:
                grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
                print(f"Mean {numeric_field_id} grouped by {group_field_id}:")
                print(grouped_df.head())
        else:
            print(f"Field {numeric_field_id} found but not numeric in the dataframe.")
else:
    print("No tabular data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
if 'filtered_df' in locals() and not filtered_df.empty and numeric_field_id:
    plt.figure(figsize=(8, 4))
    sns.histplot(filtered_df[numeric_field_id], bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id} (filtered values)")
    plt.xlabel(numeric_field_id)
    plt.show()

    if group_field_id:
        plt.figure(figsize=(10, 4))
        sns.barplot(data=filtered_df, x=group_field_id, y=numeric_field_id)
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No filtered data available for plotting.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

* In this notebook we demonstrated how to load, explore, and analyze a dataset defined by a Croissant schema using the `mlcroissant` library.
* We reviewed the available record sets, listed their fields using `@id`, and loaded tabular data for analysis.
* Basic exploratory data analysis and visualization were performed on available numeric fields.

* You can extend this analysis by investigating additional record sets, performing targeted analyses on proteins, peptide modifications, or abundance metrics, and by integrating with domain-specific bioinformatics tools for further study.
